In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
use catalog lendingclub;
create schema if not exists silver_cleaned;
use schema silver_cleaned;
select current_catalog(), current_schema();

In [0]:
customers_raw_df = spark.read.table('bronze.customers')

In [0]:
#Insert new column to capture ingested Date
customers_ingested_df = customers_raw_df.withColumn('Ingested_Date', current_timestamp())

In [0]:
#Remove Complete duplicate rows

customers_ingested_df.count()

In [0]:
customers_ingested_df.distinct().count()

In [0]:
customers_distinct_df = customers_ingested_df.distinct()

In [0]:
#Remove rows where annual_income is null

display(customers_distinct_df.filter("annual_income is null"))

In [0]:
customers_filtered_df = customers_distinct_df.filter("annual_income is not null")

In [0]:
#Convert employment_length into integers

customers_regex_df = customers_filtered_df.withColumn("employment_length", 
                                                      regexp_replace(col("employment_length"), r'\D', "").cast('integer'))

In [0]:
display(customers_regex_df)

In [0]:
#if employment_length value is null then we should take avg of employment_length then update

display(customers_regex_df.filter('employment_length is null'))

In [0]:
display(customers_regex_df.agg(avg(col('employment_length')).alias('Avg_employment')))

In [0]:
customer_agg = customers_regex_df.agg(floor(avg(col('employment_length'))).alias('Avg_employment')).collect()

In [0]:
customer_agg

In [0]:
customer_agg[0][0]

In [0]:
customer_empl_df = customers_regex_df.fillna(customer_agg[0][0], subset=['employment_length'])

In [0]:
display(customer_empl_df.filter('employment_length is null'))

In [0]:
#state code should be 2 character

display(customer_empl_df.select(col('address_state')).distinct())

In [0]:
customer_clean_df = customer_empl_df.withColumn('address_state',
                                               when(length(col('address_state')) > 2 , 'NA')
                                               .otherwise(col('address_state')))

In [0]:
display(customer_clean_df.select(col('address_state')).filter(length(col('address_state'))>2))

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/cleaned/Customer/", recurse=True)

In [0]:
customer_clean_df.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/cleaned/Customer/')

In [0]:
%sql
create or replace table silver_cleaned.customers
as
select * 
from delta.`/Volumes/lendingclub/storagelocation/cleaned/Customer/`